# W6 结果评估

以 ground truth 文件评估，评估 baselines 的 top-K 命中情况。

**评估方式**：对每个 query，看 baseline 返回的 top-K 里有多少个 **(id_t1, id_t2)** 在 ground truth 的 top-K 里（L2 距离 ≤ gt top-K 的边界）。即：在 gt 的 top-K 集合内的命中数。

In [1]:
import json
from pathlib import Path

RESULTS_DIR = Path(".")
GT_FILE = "w6_queries_100_gt.json"

def load_results(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def answer_to_pairs(answer):
    """GT answer rows are dicts {id_t1, id_t2, score} -> set of (id_t1, id_t2)."""
    return set((row["id_t1"], row["id_t2"]) for row in answer)

def query_results_to_map(data: dict) -> dict:
    """results 列表 -> query_id -> answer (list of rows)"""
    return {r["query_id"]: r["answer"] for r in data["results"]}

# Ground truth must exist
assert (RESULTS_DIR / GT_FILE).exists(), f"Ground truth file not found: {GT_FILE}"

gt_data = load_results(RESULTS_DIR / GT_FILE)
gt_by_qid = query_results_to_map(gt_data)
gt_topk_sets = {qid: answer_to_pairs(answers) for qid, answers in gt_by_qid.items()}

# Auto-discover baseline files (exclude GT), load once, append run_id to dase runs.
_files = [f for f in RESULTS_DIR.glob("*.json")
          if f.name != GT_FILE and "_gt" not in f.name]
BASELINES = []  # list of (name, filename, data) sorted by name
for _f in _files:
    _d = load_results(_f)
    _base = _d.get("baseline", _d.get("method", _f.stem))
    _rid = _d.get("run_id")
    _name = f"{_base}@{_rid}" if _rid else _base
    BASELINES.append((_name, _f.name, _d))
BASELINES.sort(key=lambda x: x[0])

print(f"Ground truth: {GT_FILE}, queries = {len(gt_topk_sets)}")
k_example = len(gt_by_qid["w6_001"])
print(f"Example w6_001: K = {k_example}")
print(f"Baselines ({len(BASELINES)}):")
for _name, _fname, _d in BASELINES:
    print(f"  {_name}  <-  {_fname}")

Ground truth: w6_queries_100_gt.json, queries = 84
Example w6_001: K = 20
Baselines (5):
  dase@20260414_170436  <-  20260414_170436.json
  dase@20260414_170614  <-  20260414_170614.json
  dase@20260414_170627  <-  20260414_170627.json
  dase@20260414_170640  <-  20260414_170640.json
  dase@20260414_170703  <-  20260414_170703.json


In [2]:
def eval_one_baseline(baseline_by_qid: dict, gt_by_qid: dict):
    """
    For each query:
      - GT has n results sorted by score a1 <= a2 <= ... <= an (n can be < K).
      - Count m = number of baseline results with score <= an (equal or better than worst GT result).
      - recall = m / n  if n > 0,  else 1.0 (both found nothing).
    """
    per_query = []
    total_hits = 0
    total_denom = 0
    for qid, answers in baseline_by_qid.items():
        if qid not in gt_by_qid:
            continue
        gt_answers = gt_by_qid[qid]
        n = len(gt_answers)
        if n == 0:
            continue
        an = max(row["score"] for row in gt_answers)  # worst (largest) score in GT
        m = sum(1 for row in answers if row[-1] <= an + 1e-6)
        recall = m / n
        per_query.append({"query_id": qid, "K": n, "hits": m, "recall": recall})
        total_hits += m
        total_denom += n
    mean_recall = total_hits / total_denom if total_denom else 0.0
    return per_query, mean_recall

In [3]:
summary = []
for name, fname, data in BASELINES:
    by_qid = query_results_to_map(data)
    per_query, mean_recall = eval_one_baseline(by_qid, gt_by_qid)
    n_queries = len(per_query)
    total_sec = data.get("total_elapsed_sec", None)
    qps = n_queries / total_sec if total_sec and total_sec > 0 else 0.0
    summary.append({
        "baseline": name,
        "mean_recall": mean_recall,
        "n_queries": len(per_query),
        "qps": qps,
    })
    print(f"{name}: mean_recall = {mean_recall:.4f}, qps = {qps:.4f}")

# also add psql (ground truth) itself
gt_n = gt_data.get("n_queries", len(gt_data.get("results", [])))
gt_sec = gt_data.get("total_elapsed_sec", None)
gt_qps = gt_n / gt_sec if gt_sec and gt_sec > 0 else 0.0
summary.append({
    "baseline": gt_data.get("baseline", gt_data.get("method", "psql")),
    "mean_recall": 1.0,
    "n_queries": gt_n,
    "qps": gt_qps,
})

dase@20260414_170436: mean_recall = 0.9958, qps = 8.7202
dase@20260414_170614: mean_recall = 0.9958, qps = 9.7681
dase@20260414_170627: mean_recall = 0.9958, qps = 9.9082
dase@20260414_170640: mean_recall = 0.9958, qps = 10.1088
dase@20260414_170703: mean_recall = 0.9958, qps = 10.3798


In [4]:
import pandas as pd

df = pd.DataFrame(summary).sort_values("mean_recall", ascending=False)
print(df.to_string(index=False))

            baseline  mean_recall  n_queries       qps
        ground_truth     1.000000         84  0.000000
dase@20260414_170436     0.995773         84  8.720166
dase@20260414_170614     0.995773         84  9.768054
dase@20260414_170627     0.995773         84  9.908204
dase@20260414_170640     0.995773         84 10.108789
dase@20260414_170703     0.995773         84 10.379787
